# Zuständigkeitsfinder · hack4GDI-DE
Die Logik steckt in `zustaendigkeit.py` (Geocoding, WFS-Cache, Punkt-in-Polygon).
Dieses Notebook zeigt die Schritte einzeln und endet bei der Funktion `finde_zustaendigkeit`,
die auch von `api.py` (HTTP-API) und `widget.html` genutzt wird.

In [ ]:
import pandas as pd
import folium
import requests
from IPython.display import display
from owslib.wfs import WebFeatureService

import zustaendigkeit as zf

LAYER = "bzb-open:ba_arbeitsagenturen"  # Zuständigkeitsbereiche der Arbeitsagenturen
ADRESSE = "Jakob-Welder-Weg 18, 55128 Mainz"  # vollständige Straßenadresse inkl. PLZ und Ort

## 3 · WFS-GetCapabilities auswerten

In [ ]:
# Capabilities mit User-Agent holen und an OWSLib übergeben (manche Server blocken Standard-Clients).
r = requests.get(
    zf.WFS_URL,
    params={"service": "WFS", "version": "1.1.0", "request": "GetCapabilities"},
    headers={"User-Agent": zf.USER_AGENT},
    timeout=60,
)
r.raise_for_status()
wfs = WebFeatureService(url=zf.WFS_URL, version="1.1.0", xml=r.content)

print("Diensttitel:", wfs.identification.title)
print("Anbieter:   ", wfs.provider.name if wfs.provider else "(nicht angegeben)")
print("\nFeature-Typen:")
for name in wfs.contents:
    print(" -", name)

print(f"\nLAYER '{LAYER}' vorhanden:", LAYER in wfs.contents)

Die Layer der übrigen Behördentypen werden in `zustaendigkeit.py` per Namenssuche in genau dieser Liste gefunden.

In [ ]:
for key in zf.BEHOERDENTYPEN:
    try:
        print(f"{key:20s} -> {zf.layer_fuer(key)}")
    except zf.LayerNichtGefunden as e:
        print(f"{key:20s} -> NICHT GEFUNDEN\n{e}")

## 4 · Zuständigkeitsflächen als GeoDataFrame laden
`lade_flaechen` holt das GeoJSON (seitenweise, falls der Server limitiert), speichert es lokal
und korrigiert vertauschte Achsen sowie fehlende CRS-Angaben automatisch.

In [ ]:
gdf = zf.lade_flaechen(LAYER)

print("Anzahl Zuständigkeitsflächen:", len(gdf))
print("Attribute:", gdf.columns.tolist())
print("CRS:", gdf.crs)
print("Bounds (minx, miny, maxx, maxy):", gdf.total_bounds.round(3))
display(gdf.drop(columns="geometry").head())

**Plausibilitätsprüfung:** Für Deutschland müssen x (Länge) zwischen ca. 5,8 und 15,1
und y (Breite) zwischen ca. 47,2 und 55,1 liegen.

## 5 · Adresse → Koordinate

In [ ]:
ort = zf.geocode(ADRESSE)
print(ort.anzeige)
print(f"lat={ort.lat:.5f}, lon={ort.lon:.5f}")
for h in ort.hinweise:
    print("Hinweis:", h)

## 6 · Punkt-in-Polygon: Welche Fläche enthält den Punkt?

In [ ]:
treffer = zf.zustaendigkeit_an_punkt(ort.lat, ort.lon, "arbeitsagentur")
for t in treffer:
    print(t["name"])
    display(pd.Series(t["attribute"]))
if len(treffer) != 1:
    print(f"Achtung: {len(treffer)} Treffer – Grenzlage oder Datenproblem.")

## 7 · Alles in einer Funktion: Adresse + Behördentypen → zuständige Stellen

In [ ]:
erg = zf.finde_zustaendigkeit(ADRESSE)  # ohne `typen`: alle Behördentypen

zeilen = []
for e in erg["ergebnisse"]:
    zeilen.append({
        "Typ": e["label"],
        "Zuständig": " | ".join(t["name"] or "?" for t in e.get("treffer", [])) or "–",
        "Hinweis/Fehler": e.get("fehler") or e.get("hinweis") or "",
    })
display(pd.DataFrame(zeilen))

## 8 · Karte

In [ ]:
erg = zf.finde_zustaendigkeit(ADRESSE, typen=["arbeitsagentur", "jobcenter"], mit_geometrie=True)
o = erg["ort"]
m = folium.Map(location=[o["lat"], o["lon"]], zoom_start=10, tiles="OpenStreetMap")
farben = {"arbeitsagentur": "#2f7b7b", "jobcenter": "#c0762b"}
for e in erg["ergebnisse"]:
    for t in e.get("treffer", []):
        folium.GeoJson(
            t["geometrie"],
            name=e["label"],
            tooltip=f'{e["label"]}: {t["name"]}',
            style_function=lambda _, c=farben.get(e["typ"], "#555"): {"color": c, "fillColor": c, "fillOpacity": 0.12, "weight": 2},
        ).add_to(m)
folium.Marker([o["lat"], o["lon"]], tooltip=o["anzeige"]).add_to(m)
folium.LayerControl().add_to(m)
m

## 9 · Belegen, wo der Geodaten-Lookup besser ist als die Suchmaschine
Das stärkste Argument für das Projekt: Adressen sammeln, bei denen die Suchmaschine eine falsche oder nur
die nächstgelegene Behörde nennt, und den Lookup dagegen stellen. Trage bei `Suchmaschine` ein, was die
erste Suche (z. B. „Arbeitsagentur <Ort>") ergibt. Besonders aussagekräftig sind Adressen in Grenzgemeinden
und in Orten, deren Agentur nicht im selben Ort sitzt.

In [ ]:
testadressen = [
    # (Adresse, Behördentyp, was die Suchmaschine als erstes nennt – selbst eintragen)
    ("Jakob-Welder-Weg 18, 55128 Mainz", "arbeitsagentur", ""),
    ("Franzensbadstraße 4, 86199 Augsburg", "arbeitsagentur", ""),
    # ("...", "arbeitsagentur", ""),
]

zeilen = []
for adr, typ, suchmaschine in testadressen:
    e = zf.finde_zustaendigkeit(adr, typen=[typ])["ergebnisse"][0]
    lookup = " | ".join(t["name"] or "?" for t in e.get("treffer", [])) or e.get("fehler", "–")
    zeilen.append({"Adresse": adr, "Typ": typ, "Geodaten-Lookup": lookup, "Suchmaschine": suchmaschine,
                   "Übereinstimmung": (suchmaschine.strip().lower() in lookup.lower()) if suchmaschine else None})
display(pd.DataFrame(zeilen))